<a href="https://colab.research.google.com/github/asomers205/DS2002FA26/blob/main/notebooks/05-etl-walmart/2026-10-16%20%E2%80%94%20Walmart%20Warm-Up%20SKU%20Mess%20%E2%80%94%20Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [1]:
import pandas as pd
from io import StringIO

PATH = '../06-midterm-walmart/data/product_catalog.csv'
RAW = 'https://raw.githubusercontent.com/asomers205/DS2002FA26/main/notebooks/06-midterm-walmart/data/product_catalog.csv'
SAMPLE = (
    'sku,product_name,category\n'
    'PT-12,pop-tarts strawberry,Breakfast Foods\n'
    'SKU#459812,Strawberry Pop-Tart,Breakfast\n'
    'POPTART-STRAW,Pop-Tarts Strawberry,Breakfast'
)

try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    # Colab only downloads this notebook, so the relative data path is missing.
    try:
        cat = pd.read_csv(RAW)
    except Exception:
        cat = pd.read_csv(StringIO(SAMPLE))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [2]:
pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print('count:', len(pop))
pop

count: 4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [3]:
strawberry = pop[pop['product_name'].str.lower().str.contains('straw', na=False)]
print('strawberry Pop-Tart rows:', len(strawberry))
display_cols = [c for c in ['sku', 'product_name', 'category', 'package_size'] if c in strawberry.columns]
print(strawberry[display_cols].to_string(index=False))

other = pop.drop(index=strawberry.index)
print('Pop-Tart rows that are not strawberry:', len(other))
other[display_cols]

strawberry Pop-Tart rows: 4
          sku         product_name        category package_size
        PT-12 pop-tarts strawberry Breakfast Foods       4-pack
   SKU#459812  Strawberry Pop-Tart       Breakfast          4pk
POPTART-STRAW Pop-Tarts Strawberry       Breakfast       6 pack
PT_STRAWBERRY pop-tarts strawberry       breakfast       4 pack
Pop-Tart rows that are not strawberry: 0


,sku,product_name,category,package_size


**Pop-Tart rows that are a different product:** none in this catalog. All four names that contain "pop" are strawberry Pop-Tarts (`PT-12`, `SKU#459812`, `POPTART-STRAW`, `PT_STRAWBERRY`). A different flavor, such as blueberry, would stay unmapped.

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [4]:
# Same canonical as Monday's lecture. .map() alone turns every other SKU into NaN;
# fillna puts the original SKU back so consolidation only renames.
CANON = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'POPTART-STRAW': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW',
}

rows_before = len(cat)
cat['canonical_sku'] = cat['sku'].map(CANON).fillna(cat['sku'])
show_cols = [c for c in ['sku', 'product_name', 'package_size', 'canonical_sku'] if c in cat.columns]
cat.loc[cat['sku'].isin(CANON), show_cols]

,sku,product_name,package_size,canonical_sku
0,PT-12,pop-tarts strawberry,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,4 pack,POPTART-STRAW


### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [5]:
print('before')
print(cat['category'].value_counts(dropna=False).sort_index().to_string())

cat['category_norm'] = cat['category'].str.strip().str.title()

print('\nafter')
print(cat['category_norm'].value_counts(dropna=False).sort_index().to_string())

print('\nStill two labels for one category after title case:')
print('- Breakfast (3) and Breakfast Foods (1)')
print('- Emergency (9) and Emergency Supplies (3)')
print('- Grocery (5) and Groceries (1)')
print('- Alcohol (1) and Alcoholic Beverages (1)')
print('- Baby (3) and Baby Products (1)')

before
category
ALCOHOL                1
Alcoholic Beverages    1
Baby                   2
Baby Products          1
Breakfast              2
Breakfast Foods        1
ELECTRONICS            2
EMERGENCY              3
ENTERTAINMENT          3
Electronics            1
Emergency              5
Emergency Supplies     3
GROCERY                3
Groceries              1
Grocery                1
HOUSEHOLD              1
Household              1
Snacks                 1
baby                   1
breakfast              1
electronics            1
emergency              1
entertainment          2
grocery                1
health                 2
household              2
snacks                 1

after
category_norm
Alcohol                1
Alcoholic Beverages    1
Baby                   3
Baby Products          1
Breakfast              3
Breakfast Foods        1
Electronics            4
Emergency              9
Emergency Supplies     3
Entertainment          5
Groceries              1
Grocery      

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [6]:
# Monday's candidate list: lowercase, strip punctuation, sort the words.
cat['name_key'] = (
    cat['product_name'].str.lower()
    .str.replace(r'[^a-z ]', '', regex=True)
    .str.split()
    .apply(lambda words: ' '.join(sorted(words)))
)

candidates = (
    cat.groupby('name_key')
    .agg(n_skus=('sku', 'size'), skus=('sku', list), names=('product_name', list))
    .loc[lambda frame: frame['n_skus'] > 1]
    .sort_values(['n_skus', 'name_key'], ascending=[False, True])
)
print('groups with more than one SKU:', len(candidates))
candidates

groups with more than one SKU: 13


,n_skus,skus,names
name_key,,,
board game monopoly,3,"[BG-MONO, MONOPOLY-01, GAME-MONO]","[Monopoly Board Game, Monopoly Board Game, Boa..."
poptarts strawberry,3,"[PT-12, POPTART-STRAW, PT_STRAWBERRY]","[pop-tarts strawberry, Pop-Tarts Strawberry, p..."
aa batteries pack,2,"[BAT-AA4, BATT_AA_4PK]","[AA Batteries 4-Pack, AA Batteries 4-Pack]"
beer light pack,2,"[BEER-24, LTBEER24]","[Light Beer (24-pack), Light Beer (24-pack)]"
bread white,2,"[BREAD-WH, WHITEBREAD]","[WHITE BREAD, WHITE BREAD]"
butter peanut,2,"[PB-16, PNUTBTR-16]","[PEANUT BUTTER, Peanut Butter]"
candles emergency,2,"[CNDL-EM, CANDLE-EMRG]","[Candles (Emergency), Emergency Candles]"
canned soup,2,"[SOUP-CAN, CNSOUP-01]","[Soup - Canned, Canned Soup]"
cards playing,2,"[CARDS-01, PLAYCARD-STD]","[Playing Cards, PLAYING CARDS]"


### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [7]:
assert cat['canonical_sku'].notna().all(), 'canonical_sku has nulls'
assert len(cat) == rows_before, 'mapping changed the row count'
unmapped = ~cat['sku'].isin(CANON)
assert (cat.loc[unmapped, 'canonical_sku'] == cat.loc[unmapped, 'sku']).all()

print('rows before/after:', rows_before, len(cat))
print('null canonical SKUs:', int(cat['canonical_sku'].isna().sum()))
print('rows renamed:', int((cat['canonical_sku'] != cat['sku']).sum()))
print('mapping is safe')

rows before/after: 45 45
null canonical SKUs: 0
rows renamed: 3
mapping is safe


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

**a)** Monopoly and 24-pack bottled water.

Monopoly is `BG-MONO` ("Monopoly Board Game"), `MONOPOLY-01` ("Monopoly Board Game"), and `GAME-MONO` ("Board Game - Monopoly"). The word-sort key `board game monopoly` already puts all three in one group. I would map them to `BG-MONO`.

Bottled water is `WTR-24` ("BOTTLED WATER 24PK"), `WATER24PK` ("Bottled Water 24-Pack"), and `WTR_24PK` ("Water - 24 Pack"). The same normalizer misses them: it turns the names into `bottled water pk`, `bottled water pack`, and `water pack`, because it drops the `24` and treats `pk` and `pack` as different words. I would still group all three as one 24-count water, canonical `WATER24PK`. `package_size` is not a safe key here — two of the rows say `1 pack` even though the name says 24.

**b)** Cleaning is when two strings are the same product and the same selling unit. It becomes a business decision when merging changes what a "unit" means, or when you cannot tell whether two rows are the same item.

I am not sure about `STREAM-01` ("Roku Streaming Stick", 16.70) and `ROKU-STD` ("Streaming Device", 16.99). The prices sit next to each other, but one name is a specific stick and the other is a generic device. They might be one product entered twice, or two different electronics. I would ask the electronics category merchant who owns that item in the product master before mapping them.